# Records and the Record family

*If you haven't read the [Getting Started tutorial](../tutorials/getting_started.ipynb), read it first — it introduces the three core objects in ProbPipe (`Distribution`, `Record`, `WorkflowFunction`) and the workflow they fit into. This notebook zooms in on the second one.*

A `Record` is ProbPipe's universal structured-value container. Every time a
distribution produces a joint sample, a likelihood consumes observed data, or a
workflow carries both parameters and diagnostics through a pipeline, the thing
that moves between functions is a `Record`. Three properties make it useful in a
JAX / probabilistic setting:

- **Named fields.** Position-based tuples are ambiguous — does `(0.8, 2.5)` mean
  `(growth_rate, carrying_capacity)` or `(carrying_capacity, growth_rate)`?
  Records force every component to carry a name that travels with the value.
- **Immutability + JAX-pytree registration.** Records are transparent to
  `jax.tree.map`, `jax.jit`, and `jax.grad`, so you can treat one like a
  single JAX value and let JAX traverse into its leaves automatically.
- **Bracket-only access.** Fields are read with `rec["growth_rate"]`, never
  `rec.growth_rate`. This keeps field names separated from the regular Python
  attribute namespace (`fields`, `name`, `replace`, ...) so a user-chosen field
  name can't collide with an API method.

The family has six members:

| Class | One or many? | Leaves |
|---|---|---|
| `Record` | one element | anything (arrays, strings, xarray, ...) |
| `NumericRecord` | one element | numeric arrays only |
| `RecordArray` | batch of elements | anything |
| `NumericRecordArray` | batch of elements | numeric arrays only |
| `RecordTemplate` | structural skeleton | any leaf spec (shape tuple or opaque `None`) |
| `NumericRecordTemplate` | structural skeleton | all-numeric leaves; adds `flat_size` |

This notebook walks through each in turn, starting with the generic `Record`
and building up to the batched numeric variant used for MCMC draws.

In [1]:
import jax
import jax.numpy as jnp
import numpy as np

from probpipe import (
    Record,
    NumericRecord,
    RecordArray,
    NumericRecordArray,
    RecordTemplate,
    NumericRecordTemplate,
)


## 1. `Record` basics

Construct a `Record` by passing named fields. Leaves can be anything —
arrays, scalars, strings, xarray objects, or nested Records. Nothing is
coerced at construction time: each leaf is stored verbatim.

In [2]:
params = Record(growth_rate=1.8, carrying_capacity=70.0)
print(params)
print("fields:", params.fields)

Record(growth_rate=1.8, carrying_capacity=70.0)
fields: ('growth_rate', 'carrying_capacity')


Fields iterate in **insertion order** — the same order the keyword arguments were passed (or the order of the input dict). Access fields with `record["name"]` — not attribute access:


In [3]:
print("growth_rate:", params["growth_rate"])
print("carrying_capacity:", params["carrying_capacity"])

growth_rate: 1.8
carrying_capacity: 70.0


Attribute access deliberately doesn't work. This separation keeps your
field names from ever colliding with the API surface:

In [4]:
try:
    params.growth_rate
except AttributeError as e:
    print("AttributeError:", e)

AttributeError: 'Record' object has no attribute 'growth_rate'


The `Record` itself has the usual dict-like protocol: `len`, `in`, iteration,
`.keys()`, `.values()`, `.items()`.

In [5]:
print("len:", len(params))
print("growth_rate in params:", "growth_rate" in params)
for name, val in params.items():
    print(f"  {name} = {val}")

len: 2
growth_rate in params: True
  growth_rate = 1.8
  carrying_capacity = 70.0


### Heterogeneous leaves

Because `Record` stores leaves verbatim, fields can mix types freely:

In [6]:
observation = Record(counts=np.array([2, 1, 3, 0, 5]), label='horseshoe', duration=10.0)
print(observation)
print("counts:", observation["counts"])
print("label:", observation["label"])

Record(counts=array(shape=(5,)), label='horseshoe', duration=10.0)
counts: [2 1 3 0 5]
label: horseshoe


### Immutability

A `Record` is immutable. You can't reassign or delete a field:

In [7]:
try:
    params["growth_rate"] = 2.0
except TypeError as e:
    print("TypeError:", e)

TypeError: 'Record' object does not support item assignment


Instead, you build new Records from old ones with `replace`, `merge`,
`without`. Each returns a new Record of the same class as `self`, so the
operation is lossless for `NumericRecord` subclasses too.

In [8]:
params2 = params.replace(growth_rate=2.0)
print("original:", params)
print("replaced:", params2)

original: Record(growth_rate=1.8, carrying_capacity=70.0)
replaced: Record(growth_rate=2.0, carrying_capacity=70.0)


In [9]:
extra = Record(phi=10.0)
combined = params.merge(extra)
print("merged:", combined)

merged: Record(growth_rate=1.8, carrying_capacity=70.0, phi=10.0)


In [10]:
reduced = combined.without("phi")
print("reduced:", reduced)

reduced: Record(growth_rate=1.8, carrying_capacity=70.0)


## 2. Nested Records

Fields can themselves be Records. This is useful when you want to group
related quantities — e.g., separating physical parameters from observation
noise in a likelihood.

In [11]:
nested = Record(physics=Record(force=1.2, mass=3.5), noise=Record(sigma=0.1))
print(nested)

Record(physics=Record(force=1.2, mass=3.5), noise=Record(sigma=0.1))


Top-level access uses the string key as usual:

In [12]:
print("physics block:", nested["physics"])

physics block: Record(force=1.2, mass=3.5)


For nested leaves, either chain bracket accesses or pass a tuple key — the
tuple form is a shortcut that reads left-to-right:

In [13]:
print("via chaining:", nested["physics"]["force"])
print("via tuple:   ", nested["physics", "force"])

via chaining: 1.2
via tuple:    1.2


`jax.tree.map` walks through nested Records automatically (they're registered
as pytrees):

In [14]:
doubled = jax.tree.map(lambda x: x * 2.0, nested)
print("doubled:", doubled)

doubled: Record(physics=Record(force=2.4, mass=7.0), noise=Record(sigma=0.2))


## 3. `NumericRecord` — the numeric-leaves specialisation

`NumericRecord` is a `Record` subclass that validates every leaf at
construction time: each must be a numeric array, a numeric scalar, or a
nested `NumericRecord`. Non-numeric leaves raise `TypeError`. Numeric
leaves are coerced to `jnp.ndarray` so downstream JAX code sees a uniform
array type.

In [15]:
theta = NumericRecord(r=1.8, K=70.0, phi=10.0)
print(theta)
print("r is jnp array:", isinstance(theta["r"], jnp.ndarray))

NumericRecord(r=Array(1.8, dtype=float32, weak_type=True), K=Array(70., dtype=float32, weak_type=True), phi=Array(10., dtype=float32, weak_type=True))
r is jnp array: True


The validation catches non-numeric leaves immediately — you can't
accidentally mix a label string in with your numeric parameters:

In [16]:
try:
    NumericRecord(r=1.8, label="horseshoe")
except TypeError as e:
    print("TypeError:", e)

TypeError: NumericRecord: field 'label' must be a numeric array, numeric scalar, or nested NumericRecord, got str


### `flatten` and `unflatten`

Because every leaf is a `jax.Array`, `NumericRecord` gains a lossless flat-vector representation. `flatten()` concatenates every leaf's raveled values in field-insertion order; `unflatten(flat, template=...)` reverses the process.


In [17]:
params = NumericRecord(r=1.8, K=jnp.array([70.0, 72.0]), phi=10.0)
flat = params.flatten()
print("flat_size:", params.flat_size)
print("flat vector:", flat)

flat_size: 4
flat vector: [ 1.8 70.  72.  10. ]


`flat_size` is cached at construction and matches `len(flatten())`. It's
the number of scalar elements across every numeric leaf — useful when
feeding the Record into an algorithm that wants a fixed-dimensional input
(MCMC samplers, optimisers, neural nets).

## 4. `RecordTemplate` / `NumericRecordTemplate` — the structural skeleton

`unflatten` needs to know the original field names and per-field shapes,
since those are erased by flattening. `RecordTemplate` carries that
structural information with no data attached. There are two closely
related classes:

- **`RecordTemplate`** — the general case. Each spec is either a shape
  tuple (numeric leaf), `None` (opaque leaf — string, label, any
  non-array data), or a nested sub-template. Exposes `leaf_shapes`.
- **`NumericRecordTemplate`** — the all-numeric specialisation. Rejects
  `None` specs outright, so every leaf is a shape tuple or a nested
  `NumericRecordTemplate`. Adds `flat_size` (total scalar count) and
  `numeric_leaf_shapes`, which are only meaningful when there's nothing
  opaque to skip.

`RecordTemplate(...)` auto-promotes to `NumericRecordTemplate` when every
spec happens to be numeric, so the common case works without naming the
subclass.

In [18]:
template = RecordTemplate.from_record(params)
print("template:", template)
print("type:", type(template).__name__)
print("leaf shapes:", template.leaf_shapes)
print("flat_size:", template.flat_size)

template: NumericRecordTemplate(r=(), K=(2,), phi=())
type: NumericRecordTemplate
leaf shapes: {'r': (), 'K': (2,), 'phi': ()}
flat_size: 4


Round-trip:

In [19]:
restored = NumericRecord.unflatten(flat, template=template)
print("restored:", restored)
print("equal to original:", restored == params)

restored: NumericRecord(r=Array(1.8, dtype=float32), K=array(shape=(2,)), phi=Array(10., dtype=float32))
equal to original: True


You can also build a template directly, without an example Record — useful
when describing the expected structure of a distribution's sample before
any draw has happened:

In [20]:
tpl = RecordTemplate(r=(), K=(2,), phi=())
print("direct template:", tpl)
print("type:", type(tpl).__name__, "(auto-promoted — all specs numeric)")
print("flat_size:", tpl.flat_size)

direct template: NumericRecordTemplate(r=(), K=(2,), phi=())
type: NumericRecordTemplate (auto-promoted — all specs numeric)
flat_size: 4


### Mixed leaves — why `flat_size` lives on the numeric template

When a template contains any opaque (`None`) leaves, it stays a plain
`RecordTemplate`. Opaque leaves don't carry a scalar count, so there
isn't a single "correct" interpretation of `flat_size` (is an opaque
leaf zero elements? one? a placeholder for something that will be
numeric later?). Rather than pick silently, the base class simply
doesn't define `flat_size` or `numeric_leaf_shapes` — those are only
defined on `NumericRecordTemplate`, where every leaf is numeric and
the count is unambiguous.

In [21]:
mixed_tpl = RecordTemplate(counts=(5,), label=None)
print("mixed template:", mixed_tpl)
print("type:", type(mixed_tpl).__name__, "(stays base — has an opaque leaf)")
print("leaf_shapes:", mixed_tpl.leaf_shapes)

# flat_size / numeric_leaf_shapes aren't defined on the mixed template.
for attr in ("flat_size", "numeric_leaf_shapes"):
    print(f"  has {attr}:", hasattr(mixed_tpl, attr))

# Trying to build a NumericRecordTemplate with an opaque leaf is rejected.
try:
    NumericRecordTemplate(counts=(5,), label=None)
except TypeError as e:
    print("NumericRecordTemplate rejects None:", e)

mixed template: RecordTemplate(counts=(5,), label=None)
type: RecordTemplate (stays base — has an opaque leaf)
leaf_shapes: {'counts': (5,), 'label': None}
  has flat_size: False
  has numeric_leaf_shapes: False
NumericRecordTemplate rejects None: NumericRecordTemplate: field 'label' is opaque (spec=None); opaque leaves are not allowed — use RecordTemplate if you need a mixed template.


### Nested templates

Templates compose: the spec for a field can be a tuple (numeric leaf),
`None` (opaque), or another template (nested structure). Auto-promotion
propagates — if every nested sub-template is numeric and the top-level
specs are all numeric, the whole tree becomes a
`NumericRecordTemplate`.

In [22]:
nested_tpl = RecordTemplate(physics=RecordTemplate(force=(), mass=()), obs=(5,))
print("nested template:", nested_tpl)
print("type:", type(nested_tpl).__name__)
print("flat_size:", nested_tpl.flat_size, "(2 scalars + 5-vector)")
print("flat leaf shapes:", nested_tpl.leaf_shapes)

nested template: NumericRecordTemplate(physics=NumericRecordTemplate(force=(), mass=()), obs=(5,))
type: NumericRecordTemplate
flat_size: 7 (2 scalars + 5-vector)
flat leaf shapes: {'physics/force': (), 'physics/mass': (), 'obs': (5,)}


## 5. `RecordArray` — batches of Records

A `RecordArray` stores a batch of Records with shared field structure.
Each field has shape `(*batch_shape, *leaf_shape)`. Typical use case: a
collection of MCMC draws, a batch of simulated datasets, a grid of
parameter proposals.

Construct from a list of Records via `.stack()`:

In [23]:
draws = [NumericRecord(r=1.8 + 0.1 * i, K=70.0 + i, phi=10.0) for i in range(4)]
batch = NumericRecordArray.stack(draws)
print("batch_shape:", batch.batch_shape)
print("template:", batch.template)
print(batch)

batch_shape: (4,)
template: NumericRecordTemplate(r=(), K=(), phi=())
NumericRecordArray(batch_shape=(4,), r=array(shape=(4,)), K=array(shape=(4,)), phi=array(shape=(4,)))


### Two kinds of indexing

**String key** → returns the batched leaf array (shape
`(*batch_shape, *leaf_shape)`):

In [24]:
print("batch['r']:", batch["r"])
print("  shape:", batch["r"].shape)

batch['r']: [1.8 1.9 2.  2.1]
  shape: (4,)


**Integer index** → returns the single element at that flat batch position,
materialised as a `Record` (or, for `NumericRecordArray`, a `NumericRecord`):

In [25]:
print("batch[2]:", batch[2])
print("  type:", type(batch[2]).__name__)

batch[2]: NumericRecord(r=Array(2., dtype=float32, weak_type=True), K=Array(72., dtype=float32, weak_type=True), phi=Array(10., dtype=float32, weak_type=True))
  type: NumericRecord


These two indexing modes never collide because Records use strings for
fields and arrays use ints for positions. You can use either without
thinking about which dimension you meant.

### `NumericRecordArray` — batched numeric operations

`NumericRecordArray` is a `RecordArray` subclass with two extra guarantees
and two extra operations. The guarantees are that every leaf must have a
numeric dtype and shape `(*batch_shape, *event_shape)`; construction
validates both. The extra operations are `flatten` / `unflatten` and
reductions.

In [26]:
print("mean across draws:", batch.mean())
print("  type:", type(batch.mean()).__name__)
print("variance across draws:", batch.var())

mean across draws: NumericRecord(r=Array(1.9499999, dtype=float32), K=Array(71.5, dtype=float32), phi=Array(10., dtype=float32))
  type: NumericRecord
variance across draws: NumericRecord(r=Array(0.0125, dtype=float32), K=Array(1.25, dtype=float32), phi=Array(0., dtype=float32))


`mean` returns a `NumericRecord` when reduction eliminates all batch
dimensions, otherwise a smaller `NumericRecordArray`. `.flatten()`
concatenates every field into a trailing axis, preserving `batch_shape`:

In [27]:
flat_batch = batch.flatten()
print("flattened shape:", flat_batch.shape)  # (4, 3) = (batch_size, flat_event_size)
print(flat_batch)

flattened shape: (4, 3)
[[ 1.8 70.  10. ]
 [ 1.9 71.  10. ]
 [ 2.  72.  10. ]
 [ 2.1 73.  10. ]]


Round-tripping through the flat representation works the same way as for
a single `NumericRecord`, but now the template describes the event
structure and the batch dimensions are preserved automatically:

In [28]:
restored_batch = NumericRecordArray.unflatten(
    flat_batch, template=batch.template, batch_shape=batch.batch_shape,
)
print("restored:", restored_batch)
print("equal:", restored_batch == batch)

restored: NumericRecordArray(batch_shape=(4,), r=array(shape=(4,)), K=array(shape=(4,)), phi=array(shape=(4,)))


equal: True


If you omit `batch_shape`, `unflatten` infers it from `flat.shape[:-1]`:

In [29]:
inferred = NumericRecordArray.unflatten(flat_batch, template=batch.template)
print("inferred batch_shape:", inferred.batch_shape)

inferred batch_shape: (4,)


### Why is `RecordArray` not hashable?

`Record` and `NumericRecord` are hashable — their `__hash__` uses the
class name, field names, and per-leaf shape+dtype, so two Records with
the same structure hash the same even though they may hold different
values. That's cheap and matches how JAX / tree-util use structural
equality.

`RecordArray` is deliberately unhashable, following NumPy's precedent:
hashing by value would require materialising the whole batch (huge for
posterior draws), and it would fail inside `jit` / `vmap` because traced
arrays aren't hashable by content. If you need a structural key, use
`(type(ra), ra.batch_shape, ra.fields, ra.template)`.

## 6. JAX interop

All four Record classes are registered as JAX pytrees with the field
names as aux data and the leaves as children. This means they flow
transparently through every JAX transform.

In [30]:
# jax.tree.map walks into the Record
theta = NumericRecord(r=1.8, K=70.0)
print("log params:", jax.tree.map(jnp.log, theta))

log params: NumericRecord(r=Array(0.5877866, dtype=float32, weak_type=True), K=Array(4.248495, dtype=float32, weak_type=True))


`jit` works because Records are pytrees — JAX knows how to trace through
them. The static structure (field names) becomes aux data, so two Records
with the same field names share a traced function:

In [31]:
@jax.jit
def logistic(theta):
    """Population at equilibrium given logistic parameters."""
    return theta["K"] * jnp.exp(theta["r"])

print("logistic(theta) =", logistic(theta))

logistic(theta) = 423.4753


Running the same jitted function on a `NumericRecordArray` broadcasts
automatically — every field is already a batched array, so JAX vectorises
the arithmetic without any extra effort:

In [32]:
theta_batch = NumericRecordArray.stack([NumericRecord(r=0.1 * i, K=70.0 + i) for i in range(5)])
print("logistic(theta_batch):", logistic(theta_batch))

logistic(theta_batch): [ 70.       78.46714  87.941    98.53969 110.39502]


## 7. `select()` for splatting into function calls

Probabilistic workflows regularly call a function that expects a handful
of named arguments with values that live inside a Record. `select()` is a
small helper that pulls specific fields out as a dict, ready to splat:

In [33]:
theta = NumericRecord(r=1.8, K=70.0, phi=10.0, unused=3.14)

def predict(r, K, x):
    return K / (1 + jnp.exp(-r * x))

x_grid = jnp.linspace(-2.0, 2.0, 5)
print("predict(**theta.select('r', 'K'), x=x_grid):")
print(predict(**theta.select("r", "K"), x=x_grid))

predict(**theta.select('r', 'K'), x=x_grid):


[ 1.8617897  9.929575  35.        60.070423  68.138214 ]


You can also rename fields on the way out by passing them as keyword
arguments — the keyword is the **new** name, the value is the existing
field name:

In [34]:
def predict_named(growth_rate, K, x):
    return K / (1 + jnp.exp(-growth_rate * x))

print("predict_named(**theta.select(growth_rate='r', K='K'), x=x_grid):")
print(predict_named(**theta.select(growth_rate="r", K="K"), x=x_grid))

predict_named(**theta.select(growth_rate='r', K='K'), x=x_grid):
[ 1.8617897  9.929575  35.        60.070423  68.138214 ]


`select()` is deliberately strict: it raises `KeyError` immediately if you
ask for a field that doesn't exist. That's the point — catching typos at
the call site, not two transforms later when a NaN mysteriously appears.

## 8. Round-tripping xarray and pandas

ProbPipe's native array form is the JAX array, but it's common to build a `Record` from richer backends — `xarray.DataArray` with dims and coords, `pandas.Series` with a `DatetimeIndex`. The aux registry in `probpipe.core._array_backend` captures backend-specific metadata so a `Record` → `NumericRecord` → `Record` round-trip restores the original objects.


In [35]:
import xarray as xr

da = xr.DataArray(
    np.array([[1.0, 2.0], [3.0, 4.0]]),
    dims=("time", "species"),
    coords={"time": [0.0, 1.0], "species": ["fox", "hare"]},
    name="counts",
)
obs = Record(counts=da)

# Forward: convert to JAX-native form. Aux is captured per leaf.
numeric = obs.to_numeric()  # NumericRecord — every leaf is jax.Array
print("numeric[counts].dtype:", numeric["counts"].dtype)
print("numeric.aux keys:    ", list((numeric.aux or {}).keys()))

# Reverse: restore each leaf to its original backend.
back = numeric.to_native()
print("back[counts]:")
print(back["counts"])


numeric[counts].dtype: float32
numeric.aux keys:     ['counts']
back[counts]:
<xarray.DataArray 'counts' (time: 2, species: 2)> Size: 16B
array([[1., 2.],
       [3., 4.]], dtype=float32)
Coordinates:
  * time     (time) float64 16B 0.0 1.0
  * species  (species) <U4 32B 'fox' 'hare'


Aux is a snapshot at construction. Any transform that rebuilds the record (`record.map`, `record.replace`, `jax.tree.map`) drops the aux — the transformed leaves are plain `jax.Array` and `to_native()` returns them as such. If you need to reattach metadata after a transform, capture the aux yourself or call `to_native()` on the pre-transform record.

Direct construction `NumericRecord(counts=da)` behaves identically to `Record(counts=da).to_numeric()` — both consult the registry, both populate `_aux`. Only the strict NumericRecord path requires every leaf to coerce via `jnp.asarray`; a generic `Record` happily holds strings, opaque objects, or anything else (use the outer `Record` for those, then call `to_numeric()` on a sub-record built from the numeric fields).

`probpipe.register_aux(...)` lets you teach the registry about additional backend types beyond the built-in xarray / pandas registrations.


## 9. Summary

- A `Record` is a named, immutable, JAX-pytree-registered container. Every structured value in ProbPipe — posterior draws, observed data, likelihood inputs — is a Record. Field iteration is insertion order; `/` is reserved as a path separator (`record["a/b/c"]` is sugar for `record["a", "b", "c"]`).
- `NumericRecord` is the strict subclass: every leaf is a `jax.Array`. `Record.to_numeric()` converts forward, `NumericRecord.to_native()` reverses; the aux registry preserves xarray / pandas metadata across the round-trip.
- `RecordArray` / `NumericRecordArray` hold a batch of Records with shared structure; they double as the output type of batched-sampling ops and the input type of batched-likelihood ops.
- `RecordTemplate` captures the structure (field names, per-field leaf shapes) without the data, so operations like `unflatten` can reverse the structure-erasing `flatten`. `NumericRecordTemplate` is the all-numeric specialisation that adds `flat_size` and `numeric_leaf_shapes`; the base-class constructor auto-promotes when every spec is numeric.
- `select()` bridges structured Records and function-call APIs that expect a handful of keyword arguments.
- Every Record class is a JAX pytree — `jit`, `grad`, and `jax.tree.map` all traverse into the leaves without special handling on your part.

The next notebook covers `@workflow_function` and broadcasting, which is how Records and Distributions get threaded together into a pipeline. `sample`, `mean`, `condition_on`, and every other ProbPipe op is a `WorkflowFunction`, so the mechanism there underpins everything else.
